# MPCount + MovingDroneCrowd++ Colab workflow

Run sections 1-8 in order for setup and inexpensive smoke tests. Full training, resume, final evaluation, and batch inference are separate guarded sections. Code and active data use fast `/content`; durable outputs use the MPCount folder in Google Drive.

In [ ]:
# ---------- CENTRAL CONFIGURATION: EDIT THIS CELL ----------
from pathlib import Path

REPO_URL = "https://github.com/gabrielxmit10/MPCount_test1.git"
REPO_REF = "main"  # branch, tag, or commit
REPO_DIR = Path("/content/MPCount_MDC")

# Reuse only the dataset archive already uploaded for the P2PNet workflow.
DRIVE_PROJECT = Path("/content/drive/MyDrive/MPCount")
DATA_SOURCE = "archive"  # archive | drive_folder | already_staged
DATASET_ARCHIVE = Path("/content/drive/MyDrive/P2PNet_MDC/data/MovingDroneCrowd++.tar")
DRIVE_DATASET_FOLDER = Path("/content/drive/MyDrive/P2PNet_MDC/data/MovingDroneCrowd++")
DATASET_ROOT = Path("/content/data/MovingDroneCrowd++")

SHANGHAITECH_WEIGHTS = DRIVE_PROJECT / "checkpoints/sta_deterministic.pth"
TRAIN_INITIALIZATION = "shanghaitech"  # shanghaitech | imagenet | random

RUN_NAME = "mpcount_mdc_run_001"
RUN_DIR = DRIVE_PROJECT / "runs" / RUN_NAME
TRAIN_SPLIT = "train.txt"
VAL_SPLIT = "val.txt"
TEST_SPLIT = "test.txt"

# Current MPCount starting values. Change deliberately after the smoke tests.
EPOCHS = 180
LEARNING_RATE = 0.001
BATCH_SIZE = 2
CROP_SIZE = 320
PATCH_SIZE = 512
NUM_WORKERS = 2

# Long-running/destructive-by-cost operations are disabled by default.
RUN_FULL_TRAINING = False
RUN_RESUME_TRAINING = False
RUN_FINAL_EVALUATION = False
RUN_BATCH_INFERENCE = False

FINAL_SPLIT = VAL_SPLIT  # use TEST_SPLIT only after choices are frozen
INFERENCE_INPUT = DATASET_ROOT / "frames/scene_1/1"
# -----------------------------------------------------------

## 1. Mount Drive and inspect the runtime

Connect a Colab GPU runtime first. This cell mounts Drive, creates only the MPCount run folder, displays the GPU, and reports fast-runtime disk space.

In [ ]:
from google.colab import drive
import datetime, json, os, platform, shlex, shutil, subprocess, sys
drive.mount("/content/drive")
RUN_DIR.mkdir(parents=True, exist_ok=True)
print("Python:", sys.version)
subprocess.run(["nvidia-smi"], check=False)
usage = shutil.disk_usage("/content")
print(f"/content free: {usage.free / 2**30:.1f} GiB")

## 2. Clone the modified MPCount repository

Push the modified repository before running this section. The real repository URL and `main` branch are already filled in. A reused runtime checkout is fast-forwarded to the requested branch.

In [ ]:
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    print("Reusing runtime checkout:", REPO_DIR)
subprocess.run(["git", "fetch", "origin"], cwd=REPO_DIR, check=True)
subprocess.run(["git", "checkout", REPO_REF], cwd=REPO_DIR, check=True)
if REPO_REF == "main":
    subprocess.run(["git", "merge", "--ff-only", "origin/main"], cwd=REPO_DIR, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements_colab.txt"], cwd=REPO_DIR, check=True)
os.chdir(REPO_DIR)

def run(command):
    command = [str(value) for value in command]
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    rendered = shlex.join(command)
    print("$", rendered)
    with (RUN_DIR / "commands.log").open("a", encoding="utf-8") as handle:
        handle.write(rendered + "\n")
    return subprocess.run(command, cwd=REPO_DIR, check=True)

def stage_shanghaitech_weights():
    if not SHANGHAITECH_WEIGHTS.is_file():
        raise FileNotFoundError(f"Upload sta_deterministic.pth first: {SHANGHAITECH_WEIGHTS}")
    local_weights = Path("/content/sta_deterministic.pth")
    if not local_weights.is_file() or local_weights.stat().st_size != SHANGHAITECH_WEIGHTS.stat().st_size:
        print("Copying ShanghaiTech weights to runtime disk...")
        shutil.copy2(SHANGHAITECH_WEIGHTS, local_weights)
    return local_weights

def smoke_initialization_args():
    if TRAIN_INITIALIZATION == "shanghaitech":
        return ["--checkpoint", stage_shanghaitech_weights()]
    if TRAIN_INITIALIZATION == "imagenet":
        return ["--pretrained"]
    if TRAIN_INITIALIZATION == "random":
        return ["--no-pretrained"]
    raise ValueError(f"Unknown TRAIN_INITIALIZATION={TRAIN_INITIALIZATION!r}")

def training_initialization_args():
    if TRAIN_INITIALIZATION == "shanghaitech":
        return ["--checkpoint", stage_shanghaitech_weights(), "--no-pretrained"]
    if TRAIN_INITIALIZATION == "imagenet":
        return ["--pretrained"]
    if TRAIN_INITIALIZATION == "random":
        return ["--no-pretrained"]
    raise ValueError(f"Unknown TRAIN_INITIALIZATION={TRAIN_INITIALIZATION!r}")

import torch, torchvision
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
print("Commit:", commit)
print("PyTorch:", torch.__version__, "torchvision:", torchvision.__version__, "CUDA:", torch.cuda.is_available())
manifest = {
    "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "repo_url": REPO_URL, "repo_ref": REPO_REF, "commit": commit,
    "dataset_archive": str(DATASET_ARCHIVE), "run_name": RUN_NAME,
    "train_initialization": TRAIN_INITIALIZATION,
    "python": platform.python_version(), "torch": torch.__version__,
    "torchvision": torchvision.__version__, "cuda_runtime": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "settings": {"epochs": EPOCHS, "learning_rate": LEARNING_RATE,
                 "batch_size": BATCH_SIZE, "crop_size": CROP_SIZE,
                 "patch_size": PATCH_SIZE, "num_workers": NUM_WORKERS},
}
(RUN_DIR / "run_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

## 3. Stage MovingDroneCrowd++ on `/content`

This reuses `/MyDrive/P2PNet_MDC/data/MovingDroneCrowd++.tar` but writes all MPCount results elsewhere. `archive` is recommended. The archive is copied once, extracted to fast runtime storage, and the temporary `/content` archive is removed. Re-running the cell reuses an already staged dataset.

In [ ]:
import tarfile
DATASET_ROOT.parent.mkdir(parents=True, exist_ok=True)
if DATASET_ROOT.exists():
    print("Dataset already staged:", DATASET_ROOT)
elif DATA_SOURCE == "archive":
    if not DATASET_ARCHIVE.is_file():
        raise FileNotFoundError(DATASET_ARCHIVE)
    archive_size = DATASET_ARCHIVE.stat().st_size
    free = shutil.disk_usage("/content").free
    if free < archive_size * 2.2:
        raise RuntimeError(f"Not enough /content disk: free={free/2**30:.1f} GiB, archive={archive_size/2**30:.1f} GiB")
    local_archive = Path("/content") / DATASET_ARCHIVE.name
    print("Copying archive to runtime disk...")
    shutil.copy2(DATASET_ARCHIVE, local_archive)
    print("Extracting...")
    with tarfile.open(local_archive) as archive:
        try:
            archive.extractall(DATASET_ROOT.parent, filter="data")
        except TypeError:
            archive.extractall(DATASET_ROOT.parent)
    local_archive.unlink()
elif DATA_SOURCE == "drive_folder":
    if not DRIVE_DATASET_FOLDER.is_dir():
        raise FileNotFoundError(DRIVE_DATASET_FOLDER)
    print("Copying dataset folder; this is slower than one archive...")
    shutil.copytree(DRIVE_DATASET_FOLDER, DATASET_ROOT)
elif DATA_SOURCE != "already_staged":
    raise ValueError(f"Unknown DATA_SOURCE={DATA_SOURCE!r}")
required = ["frames", "annotations", "train.txt", "val.txt", "test.txt"]
if not all((DATASET_ROOT / item).exists() for item in required):
    raise RuntimeError(f"Staged dataset layout is wrong: {DATASET_ROOT}")
print("Dataset ready:", DATASET_ROOT)

## 4. Validate all splits and inspect one annotated sample

The first cell validates every official mapping and saves JSON to the MPCount run directory. The second displays the first validation image with the exact head-center points consumed by MPCount.

In [ ]:
run([sys.executable, "utils/validate_mdc.py", "--root", DATASET_ROOT,
     "--splits", TRAIN_SPLIT, VAL_SPLIT, TEST_SPLIT,
     "--json-output", RUN_DIR / "dataset_validation.json"])

In [ ]:
from datasets.mdc_dataset import MDCDenClsDataset
import matplotlib.pyplot as plt
sample_set = MDCDenClsDataset(root=DATASET_ROOT, crop_size=CROP_SIZE, downsample=1,
                              method="val", unit_size=16, split_file=VAL_SPLIT)
image, points, sample_name = sample_set._load_sample(0)
plt.figure(figsize=(14, 8))
plt.imshow(image)
if len(points):
    plt.scatter(points[:, 0], points[:, 1], s=10, facecolors="none", edgecolors="lime")
plt.title(f"{sample_name} - {len(points)} heads")
plt.axis("off")
plt.show()

## 5. Cheap model/data smoke test

This uses a real MDC sample and the ShanghaiTech-A MPCount checkpoint to verify adapter tensors, batch collation, strict checkpoint loading, and one forward pass. It does not select the initialization for your future experiment.

In [ ]:
OFFICIAL_WEIGHTS = stage_shanghaitech_weights()
run([sys.executable, "smoke_test.py", "--data-root", DATASET_ROOT,
     "--checkpoint", OFFICIAL_WEIGHTS, "--device", "cuda:0",
     "--crop-size", "128"])

## 6. Verified single-image inference path

This runs the ShanghaiTech checkpoint on one real MDC validation frame, saves the count, PNG, and raw density map, then displays the visualization. It is a functionality check, not an MDC accuracy result.

In [ ]:
SMOKE_IMAGE = DATASET_ROOT / "frames/scene_4/1/1.jpg"
SMOKE_INFERENCE_DIR = RUN_DIR / "smoke_inference"
run([sys.executable, "inference.py", "--img-path", SMOKE_IMAGE,
     "--model-path", OFFICIAL_WEIGHTS, "--model-name", "final",
     "--save-path", SMOKE_INFERENCE_DIR / "counts.txt",
     "--vis-dir", SMOKE_INFERENCE_DIR, "--device", "cuda:0",
     "--patch-size", PATCH_SIZE])
from IPython.display import display
from PIL import Image
display(Image.open(next(SMOKE_INFERENCE_DIR.glob("*.png"))))

## 7. Tiny two-frame MDC evaluation smoke test

Two validation frames are counted with the ShanghaiTech checkpoint. MAE/RMSE are intentionally not meaningful yet; this verifies ground truth, tiled prediction, metrics, and persistent CSV/JSON output.

In [ ]:
run([sys.executable, "smoke_test.py", "--data-root", DATASET_ROOT,
     "--checkpoint", OFFICIAL_WEIGHTS, "--device", "cuda:0",
     "--crop-size", "128", "--tiny-eval-samples", "2",
     "--eval-output-dir", RUN_DIR / "smoke_evaluation"])

## 8. One-batch training smoke test

This uses `TRAIN_INITIALIZATION`, performs one optimizer update on a two-sample 128x128 batch, and writes a disposable checkpoint under `/content`. The default directly verifies ShanghaiTech fine-tuning. If you later choose ImageNet, set `TRAIN_INITIALIZATION = "imagenet"`, rerun the configuration and repository cells, then rerun this section before full training.

In [ ]:
TRAINING_SMOKE_CHECKPOINT = Path("/content/mpcount_training_smoke/checkpoints/latest.pth")
command = [sys.executable, "smoke_test.py", "--data-root", DATASET_ROOT,
           "--device", "cuda:0", "--crop-size", "128",
           "--train-step", "--save-checkpoint", TRAINING_SMOKE_CHECKPOINT]
command += smoke_initialization_args()
run(command)
assert TRAINING_SMOKE_CHECKPOINT.is_file()
print("Tiny training checkpoint created successfully:", TRAINING_SMOKE_CHECKPOINT)

## 9. Full training or fine-tuning

Stop here until sections 1-8 pass and you choose `TRAIN_INITIALIZATION`. Set `RUN_FULL_TRAINING = True` only when ready. MPCount validates all 724 validation frames after every epoch, saves locally for speed, and synchronizes resume/best artifacts to `RUN_DIR`.

In [ ]:
if not RUN_FULL_TRAINING:
    print("Skipped. Set RUN_FULL_TRAINING=True only after sections 1-8 pass.")
else:
    command = [sys.executable, "main.py", "--config", "configs/mdc_train.yml",
               "--task", "train", "--data-root", DATASET_ROOT,
               "--output-dir", "/content/mpcount_runs",
               "--persistent-dir", DRIVE_PROJECT / "runs",
               "--run-name", RUN_NAME, "--device", "cuda:0",
               "--num-epochs", EPOCHS, "--learning-rate", LEARNING_RATE,
               "--batch-size", BATCH_SIZE, "--crop-size", CROP_SIZE,
               "--patch-size", PATCH_SIZE, "--num-workers", NUM_WORKERS,
               "--deterministic"]
    command += training_initialization_args()
    run(command)

## 10. Resume after a Colab reset

Set `RUN_RESUME_TRAINING = True`. `EPOCHS` remains the final total epoch count, not additional epochs. Resume restores model, optimizer, OneCycle scheduler, best score, next epoch, and RNG state.

In [ ]:
LAST_RESUME = RUN_DIR / "last_resume.pth"
if not RUN_RESUME_TRAINING:
    print("Skipped. Set RUN_RESUME_TRAINING=True to resume.")
else:
    if not LAST_RESUME.is_file():
        raise FileNotFoundError(LAST_RESUME)
    run([sys.executable, "main.py", "--config", "configs/mdc_train.yml",
         "--task", "train", "--data-root", DATASET_ROOT,
         "--resume-checkpoint", LAST_RESUME, "--no-pretrained",
         "--output-dir", "/content/mpcount_runs",
         "--persistent-dir", DRIVE_PROJECT / "runs",
         "--run-name", RUN_NAME, "--device", "cuda:0",
         "--num-epochs", EPOCHS, "--learning-rate", LEARNING_RATE,
         "--batch-size", BATCH_SIZE, "--crop-size", CROP_SIZE,
         "--patch-size", PATCH_SIZE, "--num-workers", NUM_WORKERS,
         "--deterministic"])

## 11. Final full validation or held-out test

Set `RUN_FINAL_EVALUATION = True`. Keep `FINAL_SPLIT = VAL_SPLIT` while making choices; switch to `TEST_SPLIT` only once initialization and hyperparameters are frozen. Every selected frame is evaluated and a prediction CSV plus MAE/RMSE log is saved.

In [ ]:
FINAL_CHECKPOINT = RUN_DIR / "best.pth"
if not RUN_FINAL_EVALUATION:
    print("Skipped. Set RUN_FINAL_EVALUATION=True to evaluate every selected frame.")
else:
    if FINAL_SPLIT not in {VAL_SPLIT, TEST_SPLIT}:
        raise ValueError("FINAL_SPLIT must be VAL_SPLIT or TEST_SPLIT")
    if not FINAL_CHECKPOINT.is_file():
        raise FileNotFoundError(FINAL_CHECKPOINT)
    suffix = "validation_full" if FINAL_SPLIT == VAL_SPLIT else "test_full"
    evaluation_name = f"{RUN_NAME}_{suffix}"
    run([sys.executable, "main.py", "--config", "configs/mdc_test.yml",
         "--task", "test", "--data-root", DATASET_ROOT,
         "--eval-split", FINAL_SPLIT, "--checkpoint", FINAL_CHECKPOINT,
         "--output-dir", "/content/mpcount_runs",
         "--persistent-dir", DRIVE_PROJECT / "runs",
         "--run-name", evaluation_name, "--device", "cuda:0",
         "--crop-size", CROP_SIZE, "--patch-size", PATCH_SIZE,
         "--num-workers", NUM_WORKERS, "--no-pretrained", "--deterministic"])

## 12. Batch or directory inference

Choose `INFERENCE_INPUT`, then set `RUN_BATCH_INFERENCE = True`. Images are streamed one at a time. Work is written under `/content` first and copied to the current MPCount run folder afterward.

In [ ]:
INFERENCE_CHECKPOINT = RUN_DIR / "best.pth"
if not RUN_BATCH_INFERENCE:
    print("Skipped. Set RUN_BATCH_INFERENCE=True and choose INFERENCE_INPUT.")
else:
    if not INFERENCE_CHECKPOINT.is_file():
        raise FileNotFoundError(INFERENCE_CHECKPOINT)
    if not Path(INFERENCE_INPUT).exists():
        raise FileNotFoundError(INFERENCE_INPUT)
    local_output = Path("/content/mpcount_batch_inference")
    local_output.mkdir(parents=True, exist_ok=True)
    run([sys.executable, "inference.py", "--img-path", INFERENCE_INPUT,
         "--recursive", "--model-path", INFERENCE_CHECKPOINT,
         "--model-name", "final", "--save-path", local_output / "counts.txt",
         "--vis-dir", local_output, "--device", "cuda:0",
         "--patch-size", PATCH_SIZE, "--deterministic"])
    shutil.copytree(local_output, RUN_DIR / "batch_inference", dirs_exist_ok=True)
    print("Persistent inference outputs:", RUN_DIR / "batch_inference")

## 13. Logging note

The current MPCount trainer does not emit TensorBoard event files. Use `log.txt`, `resolved_config.yml`, `commands.log`, `run_manifest.json`, evaluation CSV/JSON files, checkpoints, and inference visualizations in `RUN_DIR`.

In [ ]:
print("MPCount run directory:", RUN_DIR)
print("Full training enabled:", RUN_FULL_TRAINING)
print("Resume enabled:", RUN_RESUME_TRAINING)
print("Final evaluation enabled:", RUN_FINAL_EVALUATION)
print("Batch inference enabled:", RUN_BATCH_INFERENCE)